In [1]:
# ============================================================
# TRUE SINGLE-FRAME MEDIAPIPE HEURISTIC BASELINE (RQ2)
# Rule-based classification using single-frame MediaPipe
# Face Mesh geometric features (EAR, Pitch, Yaw)
# No machine learning, no temporal modeling (T = 1)
# ============================================================

import numpy as np
from sklearn.metrics import (
    f1_score, classification_report, confusion_matrix, accuracy_score
)
from pathlib import Path

# ============================================================
# 1. CONFIGURATION
# ============================================================

T = 10  # Sample 10 frames, then select ONLY the single middle frame (index 5)
LABELS_MAP = {0: "Disengaged", 1: "Passive", 2: "Engaged"}

TEST_PATH = Path("/Users/mike/Desktop/Praxis/DAiSEE/SMOTE/features_test_master_T30.npz")

# ============================================================
# 2. LOAD PRE-EXTRACTED MEDIAPIPE FACE MESH TEST DATA
# ============================================================
# Note: "geometry" in features_test_master_T30.npz was extracted
# directly via MediaPipe Face Mesh (mp.solutions.face_mesh).

print("=" * 65)
print("SINGLE-FRAME MEDIAPIPE FACE MESH HEURISTIC BASELINE (RQ2)")
print("=" * 65)

test_data = np.load(TEST_PATH)
test_indices = np.linspace(0, 29, T, dtype=int)
test_geo = test_data["geometry"][:, test_indices, :]  # (1784, T, 10) MediaPipe features
test_labels = test_data["labels"]                     # (1784,)

print(f"Test samples:             {test_geo.shape[0]}")
print(f"MediaPipe Geometry shape: {test_geo.shape}")

# ============================================================
# 3. MEDIAPIPE FACE MESH GEOMETRIC FEATURE INDICES
# ============================================================
# Extracted via MediaPipe Face Mesh 468/478 3D landmarks:
# Index 0: ear_l    (left eye aspect ratio)
# Index 1: ear_r    (right eye aspect ratio)
# Index 2: ear_avg  (average bilateral eye aspect ratio)
# Index 3: mar      (mouth aspect ratio)
# Index 4: pitch    (head vertical tilt via solvePnP / landmarks)
# Index 5: yaw      (head horizontal rotation)
# Index 6: roll     (head roll)
# Index 7: ied      (inter-eye distance, normalized)
# Index 8: nose_x   (nose tip x coordinate, normalized)
# Index 9: nose_y   (nose tip y coordinate, normalized)

IDX_EAR_AVG = 2
IDX_PITCH = 4
IDX_YAW = 5

# ============================================================
# 4. EXTRACT SINGLE-FRAME MEDIAPIPE FEATURES (MIDDLE FRAME, T=1)
# ============================================================

middle_idx = T // 2
single_frame_mp = test_geo[:, middle_idx, :]  # (1784, 10) -> 1 frame per clip

ear_values = single_frame_mp[:, IDX_EAR_AVG]
pitch_values = np.abs(single_frame_mp[:, IDX_PITCH])
yaw_values = np.abs(single_frame_mp[:, IDX_YAW])

print(f"\nMediaPipe Single-Frame Feature Statistics (Middle Frame idx={middle_idx}):")
print(f"  EAR   — mean: {ear_values.mean():.4f}, std: {ear_values.std():.4f}, "
      f"min: {ear_values.min():.4f}, max: {ear_values.max():.4f}")
print(f"  Pitch — mean: {pitch_values.mean():.4f}, std: {pitch_values.std():.4f}, "
      f"min: {pitch_values.min():.4f}, max: {pitch_values.max():.4f}")
print(f"  Yaw   — mean: {yaw_values.mean():.4f}, std: {yaw_values.std():.4f}, "
      f"min: {yaw_values.min():.4f}, max: {yaw_values.max():.4f}")

# ============================================================
# 5. MEDIAPIPE HEURISTIC CLASSIFIERS (BINARY & TRI-STATE)
# ============================================================

def mediapipe_heuristic_binary(ear, pitch, yaw,
                               ear_threshold=0.15,
                               pitch_threshold=0.5,
                               yaw_threshold=0.5):
    """
    Standard MediaPipe geometric threshold rule (Qarbal et al., 2025):
    - Disengaged (0): Eyes closed (EAR < threshold) OR looking away (Pitch/Yaw > threshold)
    - Engaged (2): Eyes open AND facing screen (cannot distinguish Passive)
    """
    if ear < ear_threshold:
        return 0  # Disengaged (eyes closed)
    elif pitch > pitch_threshold or yaw > yaw_threshold:
        return 0  # Disengaged (looking away)
    else:
        return 2  # Engaged (default — cannot detect Passive)


def mediapipe_heuristic_tristate(ear, pitch, yaw,
                                 ear_dis_thresh=0.15,
                                 ear_pas_thresh=0.21,
                                 pitch_dis_thresh=0.5,
                                 yaw_dis_thresh=0.5,
                                 pose_pas_thresh=0.25):
    """
    3-Class MediaPipe heuristic rule (gives MediaPipe a fair chance to predict Passive):
    - Disengaged (0): Severe eye closure or large head turn
    - Passive (1): Partial eyelid droop (ear_dis <= EAR < ear_pas) OR mild off-center gaze
    - Engaged (2): Alert eyes (EAR >= ear_pas) and centered head pose
    """
    if ear < ear_dis_thresh or pitch > pitch_dis_thresh or yaw > yaw_dis_thresh:
        return 0  # Disengaged
    elif ear < ear_pas_thresh or pitch > pose_pas_thresh or yaw > pose_pas_thresh:
        return 1  # Passive (sluggish / borderline posture)
    else:
        return 2  # Engaged


# ============================================================
# 6. RUN DEFAULT MEDIAPIPE HEURISTIC ON ALL TEST SAMPLES
# ============================================================

print("\n" + "=" * 65)
print("6A. MEDIAPIPE HEURISTIC RESULTS — DEFAULT THRESHOLDS")
print("  EAR < 0.15        -> Disengaged")
print("  |Pitch| > 0.5     -> Disengaged")
print("  |Yaw| > 0.5       -> Disengaged")
print("  Otherwise         -> Engaged (Passive undetectable)")
print("=" * 65)

heuristic_preds = np.array([
    mediapipe_heuristic_binary(ear_values[i], pitch_values[i], yaw_values[i])
    for i in range(len(test_labels))
])

h_f1 = f1_score(test_labels, heuristic_preds, average="macro", zero_division=0)
h_acc = accuracy_score(test_labels, heuristic_preds)

print(f"\nMacro F1-Score: {h_f1:.4f}")
print(f"Accuracy:       {h_acc:.4f}")
print("\nClassification Report:")
print(classification_report(
    test_labels, heuristic_preds,
    target_names=["Disengaged", "Passive", "Engaged"],
    zero_division=0
))

# ============================================================
# 7. THRESHOLD SWEEP (2-CLASS & 3-CLASS MEDIAPIPE RULES)
# ============================================================

print("\n" + "=" * 65)
print("7. THRESHOLD SWEEP — BEST MEDIAPIPE HEURISTIC PERFORMANCE")
print("=" * 65)

best_f1_bin = 0
best_params_bin = {}

for ear_t in [0.05, 0.10, 0.15, 0.20, 0.25, 0.30]:
    for pitch_t in [0.2, 0.3, 0.4, 0.5, 0.6, 0.8]:
        for yaw_t in [0.2, 0.3, 0.4, 0.5, 0.6, 0.8]:
            preds = np.array([
                mediapipe_heuristic_binary(
                    ear_values[i], pitch_values[i], yaw_values[i],
                    ear_t, pitch_t, yaw_t
                )
                for i in range(len(test_labels))
            ])
            f1 = f1_score(test_labels, preds, average="macro", zero_division=0)
            if f1 > best_f1_bin:
                best_f1_bin = f1
                best_params_bin = {"ear": ear_t, "pitch": pitch_t, "yaw": yaw_t}

# Also sweep 3-class heuristic so Passive (class 1) can be predicted
best_f1_tri = 0
best_params_tri = {}

for ear_dis in [0.10, 0.15, 0.18]:
    for ear_pas in [0.20, 0.22, 0.25, 0.28]:
        for pose_dis in [0.4, 0.5, 0.6]:
            for pose_pas in [0.15, 0.20, 0.25, 0.30]:
                preds_tri = np.array([
                    mediapipe_heuristic_tristate(
                        ear_values[i], pitch_values[i], yaw_values[i],
                        ear_dis, ear_pas, pose_dis, pose_dis, pose_pas
                    )
                    for i in range(len(test_labels))
                ])
                f1_tri = f1_score(test_labels, preds_tri, average="macro", zero_division=0)
                if f1_tri > best_f1_tri:
                    best_f1_tri = f1_tri
                    best_params_tri = {
                        "ear_dis": ear_dis, "ear_pas": ear_pas,
                        "pose_dis": pose_dis, "pose_pas": pose_pas
                    }

print(f"\nBest Binary MediaPipe Heuristic Macro F1:    {best_f1_bin:.4f}")
print(f"  Thresholds: {best_params_bin}")
print(f"\nBest Tri-State MediaPipe Heuristic Macro F1: {best_f1_tri:.4f}")
print(f"  Thresholds: {best_params_tri}")

# ============================================================
# 8. RQ2 FINAL COMPARISON SUMMARY
# ============================================================

transformer_t20_f1 = 0.3546
best_mp_baseline_f1 = max(best_f1_bin, best_f1_tri)
delta_f1 = transformer_t20_f1 - best_mp_baseline_f1

print("\n" + "=" * 65)
print("RQ2 COMPARISON: SINGLE-FRAME MEDIAPIPE vs TEMPORAL TRANSFORMER")
print("=" * 65)
print(f"{'Model':<45} {'Macro F1':>10}")
print("-" * 57)
print(f"{'MediaPipe Heuristic (default thresholds)':<45} {h_f1:>10.4f}")
print(f"{'MediaPipe Heuristic (best binary sweep)':<45} {best_f1_bin:>10.4f}")
print(f"{'MediaPipe Heuristic (best 3-state sweep)':<45} {best_f1_tri:>10.4f}")
print(f"{'Logistic Regression (MediaPipe geo only)':<45} {'0.3038':>10}")
print(f"{'Logistic Regression (GhostFaceNet + MP)':<45} {'0.3379':>10}")
print(f"{'Proposed Temporal Transformer (T=10)':<45} {'0.3484':>10}")
print(f"{'Proposed Temporal Transformer (T=20)':<45} {transformer_t20_f1:>10.4f}")
print("-" * 57)
print(f"RQ2 Delta Macro-F1 (Transformer - MediaPipe): +{delta_f1:.4f} (Target >= 0.010)")
print(f"RQ2 Status: {'PASS' if delta_f1 >= 0.010 else 'FAIL'}")

SINGLE-FRAME MEDIAPIPE FACE MESH HEURISTIC BASELINE (RQ2)
Test samples:             1784
MediaPipe Geometry shape: (1784, 10, 10)

MediaPipe Single-Frame Feature Statistics (Middle Frame idx=5):
  EAR   — mean: -3.0187, std: 9.6175, min: -43.6706, max: 40.2666
  Pitch — mean: 0.3393, std: 0.0636, min: 0.0000, max: 0.4872
  Yaw   — mean: 0.3347, std: 0.0576, min: 0.0000, max: 0.4713

6A. MEDIAPIPE HEURISTIC RESULTS — DEFAULT THRESHOLDS
  EAR < 0.15        -> Disengaged
  |Pitch| > 0.5     -> Disengaged
  |Yaw| > 0.5       -> Disengaged
  Otherwise         -> Engaged (Passive undetectable)

Macro F1-Score: 0.1663
Accuracy:       0.2007

Classification Report:
              precision    recall  f1-score   support

  Disengaged       0.05      0.64      0.09        88
     Passive       0.00      0.00      0.00       882
     Engaged       0.45      0.37      0.41       814

    accuracy                           0.20      1784
   macro avg       0.17      0.34      0.17      1784
weighted